# Airbnb Paris – Vorexperiment: Semantische Relevanz (SHAP)
- Modell: SAP ConTextTab (verarbeitet Zahlen **und** Freitexte nativ, binäre Klassifikation)
- Kontext und Hintergrund stammen ausschließlich aus dem **Train-Split**; ausgewertet wird auf dem vollen Test-Split
- SHAP (KernelExplainer) je Feature; Aggregation numerisch vs. Freitext als Summe **und** Mittel je Feature

In [ ]:
import os
import numpy as np
import pandas as pd
import mlflow
import shap
import torch  # noqa: F401  (vor sap_rpt_oss laden: TORCH_LIBRARY-Doppelregistrierung vermeiden)
from sap_rpt_oss import SAP_RPT_OSS_Classifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, precision_recall_curve, auc

SEED = int(os.environ.get("SEED", 1))
TEXT_COLS = ["name", "description", "neighborhood_overview", "host_about"]
print("SEED", SEED)

## Daten & Split laden
- Numerischer Block aus dem seed-spezifischen `cleaned`, Freitexte aus `cleaned_text`

In [ ]:
base = pd.read_csv(f"../../data/preprocessed/cleaned_airbnb_paris_seed{SEED}.csv").set_index("row_id")
texts = pd.read_csv("../../data/preprocessed/cleaned_text_airbnb_paris.csv", keep_default_na=False).set_index("row_id")
split = pd.read_csv(f"../../data/splits/split_airbnb_paris_seed{SEED}.csv").set_index("row_id")

df = base.join(texts[TEXT_COLS])
y = (1 - df["is_top_rating"]).astype(int)  # 1 = Outlier
X = df.drop(columns=["is_top_rating"])
num_cols = [c for c in X.columns if c not in TEXT_COLS]
print("Zeilen:", len(X), "| Features:", X.shape[1], "| numerisch:", len(num_cols), "| Text:", len(TEXT_COLS))

## Kontext & erklärte Zeilen – nur aus Train
- Kontext 1:4 balanciert (120/480), weil ConTextTab sonst zu wenige Outlier im Kontext sieht
- Erklärte Zeilen disjunkt vom Kontext

In [ ]:
tr_idx = split.index[split["split"] == "train"]
te_idx = split.index[split["split"] == "test"]

rng = np.random.RandomState(SEED)
out_tr = y.loc[tr_idx].index[y.loc[tr_idx] == 1].to_numpy()
in_tr = y.loc[tr_idx].index[y.loc[tr_idx] == 0].to_numpy()
rng.shuffle(out_tr)
rng.shuffle(in_tr)

ctx_id = np.concatenate([out_tr[:120], in_tr[:480]])
explain_id = np.concatenate([out_tr[120:130], in_tr[480:500]])  # 10 Outlier + 20 Inlier
print("Kontext:", len(ctx_id), "| erklärt:", len(explain_id), "| Test:", len(te_idx))

## ConTextTab fitten

In [ ]:
ctx = SAP_RPT_OSS_Classifier(max_context_size=8192, bagging=8)
ctx.fit(X.loc[ctx_id], y.loc[ctx_id])  # y als Series mit X-Index -> korrektes Alignment im Modell
outlier_col = sorted(y.loc[ctx_id].unique().tolist()).index(1)

## Performance-Gate (vor SHAP)
- Voller Test-Split in **natürlicher** Verteilung; die SHAP-Analyse ist nur sinnvoll, wenn das Modell hier überzeugt

In [ ]:
y_test = y.loc[te_idx]
proba = ctx.predict_proba(X.loc[te_idx])[:, outlier_col]
pred = ctx.predict(X.loc[te_idx])

prec, rec, _ = precision_recall_curve(y_test, proba)
print(f"Test: {len(te_idx)} Zeilen | Outlier-Rate: {y_test.mean():.4f} (= AUPRC-Zufallsbaseline)")
print(f"AUPRC   = {auc(rec, prec):.4f}")
print(f"AUC-ROC = {roc_auc_score(y_test, proba):.4f}")
print(classification_report(y_test, pred, target_names=["inlier", "outlier"], digits=4, zero_division=0))
display(pd.DataFrame(confusion_matrix(y_test, pred),
                     index=["true inlier", "true outlier"], columns=["pred inlier", "pred outlier"]))

## SHAP (KernelExplainer, modell-agnostisch)
- Erklärt P(Outlier); je Feature ein SHAP-Wert, jede Freitextspalte zählt als **ein** Feature
- Hintergrund: 10 Zeilen aus dem Trainingskontext

In [ ]:
def predict_outlier(arr):
    d = pd.DataFrame(arr, columns=X.columns)
    d[num_cols] = d[num_cols].astype(float)
    return ctx.predict_proba(d)[:, outlier_col]

background = X.loc[ctx_id].sample(10, random_state=SEED)
sv = shap.KernelExplainer(predict_outlier, background).shap_values(X.loc[explain_id], nsamples=100)
imp = pd.Series(np.abs(np.array(sv)).reshape(-1, X.shape[1]).mean(axis=0), index=X.columns)

## Ergebnis: SHAP je Feature + numerisch vs. Freitext
- Neben der Summe auch das **Mittel je Feature**, weil die Gruppen unterschiedlich groß sind

In [ ]:
display(imp.sort_values(ascending=False).round(5).to_frame("mean_abs_shap"))
numeric_total, text_total = float(imp[num_cols].sum()), float(imp[TEXT_COLS].sum())
numeric_mean, text_mean = numeric_total / len(num_cols), text_total / len(TEXT_COLS)
print(f"Summe      numerisch={numeric_total:.5f}  freitext={text_total:.5f}")
print(f"je Feature numerisch={numeric_mean:.5f}  freitext={text_mean:.5f}")

## Loggen

In [ ]:
mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("airbnb_paris_preliminary_experiment")
with mlflow.start_run(run_name="shap_contexttab"):
    mlflow.log_param("seed", SEED)
    # log every feature, prefix encodes the type (text_ vs num_)
    for c in X.columns:
        mlflow.log_metric(("text_" if c in TEXT_COLS else "num_") + c, float(imp[c]))
    mlflow.log_metric("numeric_total", numeric_total)
    mlflow.log_metric("text_total", text_total)
    mlflow.log_metric("numeric_mean", numeric_mean)
    mlflow.log_metric("text_mean", text_mean)